
# Comparison of three gene-mapping strategies in SAMap

**BioHackathon Europe 2026 — Project 23, WP2**

**Contributor:** Nada Orsulics

## Objective

Compare cross-species cell-type mapping scores obtained using:

1. BLAST (map1)
2. MMseqs2 (map2)
3. BLAST + OrthoFinder (map3)

The analysis focuses on mouse (*Mus musculus*) and zebrafish
(*Danio rerio*).

**Status:** Preliminary analysis. Results will be updated after
full-dataset runs and potentially extended to human datasets.

This notebook compares precomputed SAMap MappingT matrices.
It does not rerun SAMap or benchmark sequence-alignment runtime.



## 1. Configuration

Set `RESULTS_DIR` to the directory containing the three
MappingT files.

The default path is for the contributor's local Ubuntu/WSL
environment. Other users should change this path.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import os

RESULTS_DIR = Path(
    os.environ.get("SAMAP_RESULTS_DIR", "WP2/Orthofinder")
).expanduser().resolve()

if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(
        f"MappingT directory not found: {RESULTS_DIR}. "
        "Set SAMAP_RESULTS_DIR to the directory containing the MappingT files."
    )

FILES = {
    "BLAST": "MappingT_mm_dr_annot.csv",
    "MMseqs2": "MappingT_mm_dr_annot_map2.csv",
    "OrthoFinder": "MappingT_mm_dr_annot_orthofinder.csv",
}

OUTPUT_DIR = Path("comparison_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



## 2. Load and validate MappingT matrices

BLAST and MMseqs2 files are tab-separated matrices
with column labels but no row labels.

For these files, row labels are reconstructed from column
labels, assuming the original SAMap export uses identical
row and column ordering.

The OrthoFinder file is comma-separated and includes
explicit row labels.

All three matrices must have identical ordered cell-type
labels before comparison.


In [ ]:

def load_mapping_matrix(path, method):
    if method == "OrthoFinder":
        df = pd.read_csv(path, index_col=0)
    else:
        df = pd.read_csv(path, sep="\t")
        if df.shape[0] != df.shape[1]:
            raise ValueError(f"{method}: non-square matrix")
        df.index = df.columns.copy()

    if df.shape[0] != df.shape[1]:
        raise ValueError(f"{method}: non-square matrix")

    if df.index.has_duplicates or df.columns.has_duplicates:
        raise ValueError(f"{method}: duplicate cell-type labels")

    if df.isna().any().any():
        raise ValueError(f"{method}: missing values")

    df = df.apply(pd.to_numeric, errors="raise")
    if not np.isfinite(df.to_numpy()).all():
        raise ValueError(f"{method}: non-finite values")

    return df


matrices = {
    method: load_mapping_matrix(RESULTS_DIR / filename, method)
    for method, filename in FILES.items()
}

reference = matrices["OrthoFinder"].columns.tolist()

for method, df in matrices.items():
    assert df.index.tolist() == reference, method
    assert df.columns.tolist() == reference, method
    print(method, df.shape, "validated")



## 3. Cross-species mapping scores

Only mouse–zebrafish pairs are compared.

The within-species blocks are excluded. Both cross-species
directions are retained for matrix-level comparisons.


In [ ]:

labels = matrices["BLAST"].index

mouse = [x for x in labels if x.startswith("mm_")]
zebrafish = [x for x in labels if x.startswith("dr_")]

assert len(mouse) > 0 and len(zebrafish) > 0
assert len(mouse) + len(zebrafish) == len(labels)

pairs = (
    [(m, d) for m in mouse for d in zebrafish]
    + [(d, m) for d in zebrafish for m in mouse]
)

scores = pd.DataFrame({
    method: [df.loc[a, b] for a, b in pairs]
    for method, df in matrices.items()
})

scores.index = pd.MultiIndex.from_tuples(
    pairs, names=["source_cell_type", "target_cell_type"]
)

print("Directed cross-species comparisons:", len(scores))
display(scores.head())



## 4. Pairwise method comparison

Spearman correlation evaluates agreement in score rankings.

Mean absolute error (MAE) quantifies absolute differences
between MappingT scores.

These metrics describe method agreement, not biological
accuracy or superiority.


In [ ]:

from itertools import combinations

summary = []

for a, b in combinations(matrices, 2):
    x = scores[a].to_numpy()
    y = scores[b].to_numpy()

    rho, pvalue = spearmanr(x, y)
    mae = np.mean(np.abs(x - y))

    summary.append({
        "method_a": a,
        "method_b": b,
        "spearman_rho": rho,
        "spearman_p": pvalue,
        "MAE": mae,
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

summary_df.to_csv(
    OUTPUT_DIR / "method_comparison_summary.csv",
    index=False
)



## 5. MappingT heatmaps

The same cell-type ordering and color scale are used
for all three methods.


In [ ]:

vmax = max(df.to_numpy().max() for df in matrices.values())

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (method, df) in zip(axes, matrices.items()):
    im = ax.imshow(df.to_numpy(), vmin=0, vmax=vmax)
    ax.set_title(method)
    ax.set_xticks(range(len(df.columns)))
    ax.set_xticklabels(df.columns, rotation=90, fontsize=6)
    ax.set_yticks(range(len(df.index)))
    ax.set_yticklabels(df.index, fontsize=6)

fig.colorbar(im, ax=axes, shrink=0.6, label="MappingT score")
fig.savefig(
    OUTPUT_DIR / "mappingT_heatmaps.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()



## 6. Best cross-species matches

For each source cell type, identify the highest-scoring
target cell type in the other species.

Ties are resolved using the original matrix column order.
Top-1 agreement is descriptive and does not establish
biological correctness.


In [ ]:

top_matches = []

for source in labels:
    targets = zebrafish if source.startswith("mm_") else mouse

    row = {"source_cell_type": source}

    for method, df in matrices.items():
        values = df.loc[source, targets]
        row[f"{method}_top1"] = values.idxmax()
        row[f"{method}_score"] = values.max()

    row["all_three_agree"] = (
        row["BLAST_top1"]
        == row["MMseqs2_top1"]
        == row["OrthoFinder_top1"]
    )

    top_matches.append(row)

top_df = pd.DataFrame(top_matches)
display(top_df)

top_df.to_csv(
    OUTPUT_DIR / "top1_matches.csv",
    index=False
)



## 7. Limitations and future work

- This notebook analyzes precomputed MappingT scores.
- BLAST/MMseqs2 row labels are reconstructed from column
  order and should be verified against the SAMap export.
- Results depend on the underlying datasets, annotations,
  SAMap settings and gene-map construction.
- The two mapping directions are related observations;
  nominal correlation p-values should not be interpreted
  as independent biological replication.
- Full-dataset and human-inclusive analyses are planned.
- Method agreement alone does not identify the biologically
  most accurate gene-mapping strategy.



## 8. Top-3 cross-species match stability

For each source cell type, identify the three highest-scoring
target cell types in the other species.

Compare the overlap of Top-3 target sets between methods.
Results describe ranking stability, not biological accuracy.


In [ ]:

from itertools import combinations

top3_rows = []

for source in labels:
    targets = zebrafish if source.startswith("mm_") else mouse

    top3 = {
        method: df.loc[source, targets].nlargest(3).index.tolist()
        for method, df in matrices.items()
    }

    row = {"source_cell_type": source}

    for method, matches in top3.items():
        row[f"{method}_top3"] = "; ".join(matches)

    for a, b in combinations(matrices, 2):
        overlap = len(set(top3[a]) & set(top3[b]))
        row[f"{a}_vs_{b}_overlap"] = overlap

    row["all_three_common"] = len(
        set.intersection(*(set(v) for v in top3.values()))
    )

    top3_rows.append(row)

top3_df = pd.DataFrame(top3_rows)
display(top3_df)

top3_df.to_csv(
    OUTPUT_DIR / "top3_stability.csv",
    index=False
)

print("\nMean Top-3 overlap:")
for a, b in combinations(matrices, 2):
    col = f"{a}_vs_{b}_overlap"
    print(f"{a} vs {b}: {top3_df[col].mean():.2f} / 3")



## 9. Pairwise MappingT difference heatmaps

These heatmaps show signed differences in MappingT scores
between gene-mapping methods.

A positive value means that the first method produced a
higher MappingT score than the second method.

All plots use a common, symmetric color scale.
Differences are descriptive and do not establish
biological accuracy.


In [ ]:

from itertools import combinations
from matplotlib.colors import TwoSlopeNorm

method_pairs = list(combinations(matrices, 2))

differences = {
    (a, b): matrices[a] - matrices[b]
    for a, b in method_pairs
}

max_abs = max(
    np.abs(df.to_numpy()).max()
    for df in differences.values()
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs)

for ax, ((a, b), df) in zip(axes, differences.items()):
    im = ax.imshow(df.to_numpy(), cmap="RdBu_r", norm=norm)
    ax.set_title(f"{a} − {b}")
    ax.set_xticks(range(len(df.columns)))
    ax.set_xticklabels(df.columns, rotation=90, fontsize=6)
    ax.set_yticks(range(len(df.index)))
    ax.set_yticklabels(df.index, fontsize=6)

fig.colorbar(
    im, ax=axes, shrink=0.6,
    label="MappingT score difference"
)

fig.savefig(
    OUTPUT_DIR / "mappingT_difference_heatmaps.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
